In [1]:
import numpy as np 
import pandas as pd

In [2]:
df=pd.read_csv("./diabetes.csv")
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
df.corr()["Outcome"]

Pregnancies                 0.221898
Glucose                     0.466581
BloodPressure               0.065068
SkinThickness               0.074752
Insulin                     0.130548
BMI                         0.292695
DiabetesPedigreeFunction    0.173844
Age                         0.238356
Outcome                     1.000000
Name: Outcome, dtype: float64

In [4]:
X=df.iloc[:,:-1].values
y=df.iloc[:,-1].values

In [5]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()

In [6]:
X=scaler.fit_transform(X)

In [7]:
X.shape

(768, 8)

In [8]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=1)

In [46]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Dropout  

In [10]:
model=Sequential()
model.add(Dense(32,activation="relu",input_dim=8))
model.add(Dense(1,activation="sigmoid"))


model.compile(optimizer="Adam",loss="binary_crossentropy",metrics=["accuracy"])

/Users/kristalshrestha/Documents/Code/DeepLearning_practice/venv/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
model.fit(X_train,y_train,batch_size=32,epochs=10,validation_data=(X_test,y_test))

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6596 - loss: 0.6847 - val_accuracy: 0.6494 - val_loss: 0.6511
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6873 - loss: 0.6131 - val_accuracy: 0.7143 - val_loss: 0.5864
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7166 - loss: 0.5670 - val_accuracy: 0.7468 - val_loss: 0.5458
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7459 - loss: 0.5370 - val_accuracy: 0.7662 - val_loss: 0.5199
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7557 - loss: 0.5165 - val_accuracy: 0.7727 - val_loss: 0.5015
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7573 - loss: 0.5023 - val_accuracy: 0.7727 - val_loss: 0.4888
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7687 - loss: 0.4910 - val_accuracy: 0.7727 - val_loss: 0.4797
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7752 - loss: 0.4823 - val_accuracy: 0.7727 - val_loss:

In [12]:
# 1.How to select appropriate optimizer
# 2. How to select no of nodes in a layer
#3.How to select no. of layers
# 4. All in all one model

In [13]:
# pip install tensorboard
# pip install keras-tuner


# 1.How to select appropriate optimizer

In [14]:
import keras_tuner as kt

In [15]:
def build_model(hp):

    model=Sequential()
    model.add(Dense(32,activation="relu",input_dim=8))
    model.add(Dense(1,activation="sigmoid"))

    optimizer=hp.Choice("optimizer",values=["adam","sgd","rmsprop","adamdelta"])
    model.compile(optimizer=optimizer,loss="binary_crossentropy",metrics=["accuracy"])


    return model

In [16]:
tuner=kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=5,
    directory="HyperParameterTuningDirectory",
    project_name="1TuningOptimizer"
    )

In [17]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 4 Complete [00h 00m 00s]

Best val_accuracy So Far: 0.8051947951316833
Total elapsed time: 00h 00m 03s


In [18]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

In [19]:
model=tuner.get_best_models(num_models=1)[0]

/Users/kristalshrestha/Documents/Code/DeepLearning_practice/venv/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:794: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [20]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7704 - loss: 0.5223 - val_accuracy: 0.7922 - val_loss: 0.5132
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7736 - loss: 0.5054 - val_accuracy: 0.7857 - val_loss: 0.5023
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7752 - loss: 0.4954 - val_accuracy: 0.7857 - val_loss: 0.4938
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7736 - loss: 0.4878 - val_accuracy: 0.7987 - val_loss: 0.4872
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7736 - loss: 0.4820 - val_accuracy: 0.7922 - val_loss: 0.4818
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7720 - loss: 0.4769 - val_accuracy: 0.7922 - val_loss: 0.4785
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7736 - loss: 0.4733 - val_accuracy: 0.7922 - val_loss: 0.4762
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7720 - loss: 0.4705 - val_accuracy: 0.792

# 1.Using Best Optimizer from above result and then 
# 2. finding How to select no of nodes in a layer

In [ ]:
def build_model(hp):
    model=Sequential()
    units=hp.Int("units",min_value=8,max_value=128,step=8) # or just remove step to have step =1
    model.add(Dense(units=units,activation="relu",input_dim=8))
    model.add(Dense(1,activation="sigmoid"))

    model.compile(optimizer="rmsprop",loss="binary_crossentropy",metrics=["accuracy"])

    return model

In [23]:
tuner=kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=5,
    directory="HyperParameterTuningDirectory",
    project_name="2No_of_nodes_tuning"
    )

In [25]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 5 Complete [00h 00m 01s]
val_accuracy: 0.7727272510528564

Best val_accuracy So Far: 0.8051947951316833
Total elapsed time: 00h 00m 19s


In [27]:
tuner.get_best_hyperparameters()[0].values

{'units': 112}

In [28]:
model=tuner.get_best_models(num_models=1)[0]

/Users/kristalshrestha/Documents/Code/DeepLearning_practice/venv/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/Users/kristalshrestha/Documents/Code/DeepLearning_practice/venv/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:794: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [30]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8225 - loss: 0.3909 - val_accuracy: 0.8052 - val_loss: 0.4679
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8192 - loss: 0.3896 - val_accuracy: 0.8117 - val_loss: 0.4634
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8274 - loss: 0.3882 - val_accuracy: 0.7922 - val_loss: 0.4601
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8208 - loss: 0.3895 - val_accuracy: 0.7987 - val_loss: 0.4605
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8274 - loss: 0.3881 - val_accuracy: 0.8052 - val_loss: 0.4644
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8225 - loss: 0.3886 - val_accuracy: 0.8117 - val_loss: 0.4648
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8192 - loss: 0.3883 - val_accuracy: 0.8182 - val_loss: 0.4647
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8322 - loss: 0.3871 - val_accuracy: 0.805

# 3.How to select no. of layers

In [31]:
def build_model(hp):
    model=Sequential()

    model.add(Dense(72,activation="relu",input_dim=8))

    for i in range(hp.Int("num_layers",min_value=1,max_value=10)):
        model.add(Dense(72,activation="relu"))
    
    model.add(Dense(1,activation="sigmoid"))

    model.compile(optimizer="rmsprop",loss="binary_crossentropy",metrics=["accuracy"])

    return model

In [32]:
tuner=kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=3,
    directory="HyperParameterTuningDirectory",
    project_name="3No_ofLayersTuning"
    )

/Users/kristalshrestha/Documents/Code/DeepLearning_practice/venv/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [33]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 3 Complete [00h 00m 01s]
val_accuracy: 0.7857142686843872

Best val_accuracy So Far: 0.798701286315918
Total elapsed time: 00h 00m 04s


In [34]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 6}

In [35]:
model=tuner.get_best_models(num_models=1)[0]

/Users/kristalshrestha/Documents/Code/DeepLearning_practice/venv/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:794: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [36]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7769 - loss: 0.4554 - val_accuracy: 0.7792 - val_loss: 0.4714
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8013 - loss: 0.4248 - val_accuracy: 0.8052 - val_loss: 0.4594
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8208 - loss: 0.4072 - val_accuracy: 0.7403 - val_loss: 0.5662
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8127 - loss: 0.4180 - val_accuracy: 0.7857 - val_loss: 0.4720
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8192 - loss: 0.3903 - val_accuracy: 0.7922 - val_loss: 0.4525
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8274 - loss: 0.3860 - val_accuracy: 0.7922 - val_loss: 0.4772
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8192 - loss: 0.3788 - val_accuracy: 0.7792 - val_loss: 0.4919
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8274 - loss: 0.3667 - val_accuracy: 0.772

# final all in one

In [50]:
def build_model(hp):
    model=Sequential()
    counter=0
    for i in range(hp.Int("num_layers",min_value=1,max_value=10)):

        if counter==0:
            model.add(
                Dense(
                    hp.Int("units" + str(i),min_value=8,max_value=128,step=8),
                    activation=hp.Choice("activation" + str(i),values=["relu","tanh","sigmoid"]),
                    input_dim=8
                    )
                )
            model.add(Dropout(hp.Choice("dropout"+str(i),values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
        
        else:
            model.add(
                Dense(
                    hp.Int("units" + str(i),min_value=8,max_value=128,step=8),
                    activation=hp.Choice("activation" + str(i),values=["relu","tanh","sigmoid"]),
                    )
         
               )
            model.add(Dropout(hp.Choice("dropout"+str(i),values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
        
        counter=counter+1


    model.add(Dense(1,activation="sigmoid"))


    model.compile(optimizer=hp.Choice("optimizer",values=["rmsprop","adam","sgd","nadam","adadelta"]),
                 loss="binary_crossentropy",metrics=["accuracy"])
                
    return model

In [51]:
tuner=kt.RandomSearch(build_model,
                     objective="val_accuracy",
                     max_trials=3,
                     directory="HyperParameterTuningDirectory",
                     project_name="4FInalTUNING_EVERYTHINGATONCE_WithDropout"
                     )

In [52]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 3 Complete [00h 00m 01s]
val_accuracy: 0.48051947355270386

Best val_accuracy So Far: 0.7792207598686218
Total elapsed time: 00h 00m 05s


In [53]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 3,
 'units0': 56,
 'activation0': 'tanh',
 'dropout0': 0.4,
 'optimizer': 'nadam',
 'units1': 88,
 'activation1': 'tanh',
 'dropout1': 0.2,
 'units2': 56,
 'activation2': 'tanh',
 'dropout2': 0.2,
 'units3': 112,
 'activation3': 'tanh',
 'dropout3': 0.5,
 'units4': 72,
 'activation4': 'sigmoid',
 'dropout4': 0.1,
 'units5': 104,
 'activation5': 'relu',
 'dropout5': 0.3,
 'units6': 24,
 'activation6': 'tanh',
 'dropout6': 0.2,
 'units7': 32,
 'activation7': 'relu',
 'dropout7': 0.4,
 'units8': 32,
 'activation8': 'relu',
 'dropout8': 0.6}

In [54]:
model=tuner.get_best_models(num_models=1)[0]

/Users/kristalshrestha/Documents/Code/DeepLearning_practice/venv/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:794: UserWarning: Skipping variable loading for optimizer 'nadam', because it has 2 variables whereas the saved optimizer has 19 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [55]:
model.fit(X_train,y_train,epochs=200,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7410 - loss: 0.5216 - val_accuracy: 0.7922 - val_loss: 0.4732
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7329 - loss: 0.5096 - val_accuracy: 0.7727 - val_loss: 0.4726
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7296 - loss: 0.5104 - val_accuracy: 0.7857 - val_loss: 0.4676
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7394 - loss: 0.5125 - val_accuracy: 0.7792 - val_loss: 0.4800
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7573 - loss: 0.4850 - val_accuracy: 0.7792 - val_loss: 0.4677
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7410 - loss: 0.5213 - val_accuracy: 0.7857 - val_loss: 0.4615
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7541 - loss: 0.4965 - val_accuracy: 0.7792 - val_loss: 0.4672
Epoch 14/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7590 - loss: 0.4889 - val_accuracy: 0.792